# BPE Training Test on Valid Data

In [1]:
import sys
sys.path.insert(0, "..")

import regex as re
from cs336_basics.bpe_train import GPT2_PRETOKENIZER_PATTERN, _special_token_split_pattern

## 1. Test special token split pattern

In [2]:
text = "Hello world endoftext This is a test endoftext Goodbye"
special_tokens = ["endoftext"]
pattern = _special_token_split_pattern(special_tokens)

print("Pattern:", pattern)
print("split() result:", pattern.split(text))
print()

print("finditer() results:")
start = 0
for match in pattern.finditer(text):
    segment = text[start:match.start()]
    print(f"  Segment: {segment!r}")
    print(f"  Special token: {match.group()!r}")
    start = match.end()
print(f"  Last segment: {text[start:]!r}")

Pattern: regex.Regex('endoftext', flags=regex.V0)
split() result: ['Hello world ', ' This is a test ', ' Goodbye']

finditer() results:
  Segment: 'Hello world '
  Special token: 'endoftext'
  Segment: ' This is a test '
  Special token: 'endoftext'
  Last segment: ' Goodbye'


## 2. Read valid data

In [3]:
from pathlib import Path

valid_path = Path("../data/TinyStoriesV2-GPT4-valid.txt")
text = valid_path.read_text(encoding="utf-8")
print(f"Valid data size: {len(text):,} chars")
print(f"First 500 chars: {text[:500]!r}")

Valid data size: 22,493,387 chars
First 500 chars: 'u don\'t have to be scared of the loud dog, I\'ll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.\n<|endoftext|>\nOnce upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball. He was very sad.\nTom asked his friend, Sam, to help him search for the ball. They looked high a'


## 3. Test _pretoken_counts with finditer approach

In [4]:
from collections import Counter

pretoken_pattern = re.compile(GPT2_PRETOKENIZER_PATTERN)
special_tokens = ["endoftext"]
split_pattern = _special_token_split_pattern(special_tokens)

# Current split approach (special tokens are lost)
counts_split: Counter = Counter()
segments = split_pattern.split(text) if split_pattern is not None else [text]
print(f"Number of segments from split(): {len(segments)}")
for segment in segments:
    for match in pretoken_pattern.finditer(segment):
        pretoken_bytes = match.group().encode("utf-8")
        counts_split[tuple(bytes([b]) for b in pretoken_bytes)] += 1
print(f"Total pretokens (split): {sum(counts_split.values()):,}")
print()

Number of segments from split(): 27631
Total pretokens (split): 5,474,335



In [5]:
# finditer approach (correct, same as tokenizer.py)
counts_finditer: Counter = Counter()
if split_pattern is not None:
    start = 0
    for match in split_pattern.finditer(text):
        for pretoken_match in pretoken_pattern.finditer(text[start:match.start()]):
            pretoken_bytes = pretoken_match.group().encode("utf-8")
            counts_finditer[tuple(bytes([b]) for b in pretoken_bytes)] += 1
        start = match.end()
    for pretoken_match in pretoken_pattern.finditer(text[start:]):
        pretoken_bytes = pretoken_match.group().encode("utf-8")
        counts_finditer[tuple(bytes([b]) for b in pretoken_bytes)] += 1
else:
    for pretoken_match in pretoken_pattern.finditer(text):
        pretoken_bytes = pretoken_match.group().encode("utf-8")
        counts_finditer[tuple(bytes([b]) for b in pretoken_bytes)] += 1

print(f"Total pretokens (finditer): {sum(counts_finditer.values()):,}")
print()
print("Results match:", counts_split == counts_finditer)

Total pretokens (finditer): 5,474,335

Results match: True


## 4. Run actual BPE training on valid data

In [6]:
from cs336_basics.bpe_train import train_bpe
import time

start_time = time.time()
vocab, merges = train_bpe(
    input_path=str(valid_path),
    vocab_size=500,
    special_tokens=["endoftext"],
)
elapsed = time.time() - start_time

print(f"Training completed in {elapsed:.2f}s")
print(f"Vocab size: {len(vocab)}")
print(f"Number of merges: {len(merges)}")
print(f"\nFirst 10 merges:")
for i, (left, right) in enumerate(merges[:10]):
    print(f"  {i}: {left!r} + {right!r} -> {(left + right)!r}")

Training completed in 6.03s
Vocab size: 500
Number of merges: 243

First 10 merges:
  0: b' ' + b't' -> b' t'
  1: b'h' + b'e' -> b'he'
  2: b' ' + b'a' -> b' a'
  3: b' ' + b's' -> b' s'
  4: b' ' + b'w' -> b' w'
  5: b' t' + b'he' -> b' the'
  6: b'n' + b'd' -> b'nd'
  7: b'e' + b'd' -> b'ed'
  8: b' ' + b'b' -> b' b'
  9: b' t' + b'o' -> b' to'
